# Toy example

A fast, runnable version of the paper's pipeline on one small village (Goeschenen), using the real pipeline code from `source_code/simulation/src/`.

Real: road network, SUMO config, routes, exit speeds, the traffic simulation, the emission model, the diffusion model (same `dispersion_runner` helpers as `run_sim.py`) and the AQI post-processing.

Synthetic: the resident population. The real SynPop survey data behind the paper cannot be published, so `generate_fake_population.py` invents a population instead: random coordinates in the village plus category proportions (age, employment, car ownership) picked by hand. It matches the CSV schema the pipeline expects, so the same `get_people_data()` function reads it unchanged. The residents are used for the final exposure step (AQI read off at each resident's grid cell); they do not generate any traffic.

Left out: random local traffic, rerouting, and the house-outline plots (these need real building outlines or an internet geocoder; this notebook runs offline).

For the paper's actual results, see `../reproducibility/`. For the full pipeline source, see `../source_code/`.

Requirements: `pip install -r requirements.txt`. Run this notebook from the `toy_example/` folder.


In [ ]:
import sys, os, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TOY_DIR = os.getcwd()
print("Working in:", TOY_DIR)


## Step 1: generate a fake population

Loads the road network to get the village bounding box, then places random synthetic residents inside it.

In [ ]:
result = subprocess.run([sys.executable, "generate_fake_population.py", "500", "fake_population.csv"],
                         capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("generate_fake_population.py failed")

pop = pd.read_csv("fake_population.csv")
print(pop.shape, "synthetic residents")
pop.head()


In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(pop["xcoord"], pop["ycoord"], s=4, alpha=0.5)
ax.set_title("Synthetic resident locations (Goeschenen)")
ax.set_xlabel("SUMO x [m]"); ax.set_ylabel("SUMO y [m]")
ax.set_aspect("equal")
plt.show()


## Step 2: run the toy simulation

Same steps as `run_sim.py`: load population, step SUMO, read emissions, diffuse, save hourly snapshots, then compute the AQI and read it off at each resident.

Goeschenen is very quiet (about 170 vehicles in the first 10 h), so concentrations only rise slightly above background. We simulate 10 hours, which takes about 2 to 3 minutes. The AQI step needs 1 h of history for NO2, 8 h for CO and 24 h for PM2.5, so a 10 h run reports CO and NO2 only; pass 24 for the full three-pollutant index (~6 minutes).


In [ ]:
result = subprocess.run([sys.executable, "run_toy_simulation.py", "10.0"],
                         capture_output=True, text=True)
print(result.stdout[-1500:])
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError("run_toy_simulation.py failed")


## Step 3: look at the result

Each snapshot is `data_<pollutant>_<z-layer>_<time>.csv`, in mg/m^3; the AQI and the plot below use the exposure layer 1 (0.3-1.2 m). Because the village is quiet, absolute maps look uniform, so we plot the **excess over the ambient background** (from `config.py`) in the last snapshot.


In [ ]:
import glob
sys.path.insert(0, os.path.join(TOY_DIR, "..", "source_code", "simulation", "src"))
import config as cfg

def latest_snapshot_time(data_dir):
    times = set()
    for f in glob.glob(os.path.join(data_dir, f"data_co_{cfg.EXPOSURE_LAYER}_*.csv")):
        t = os.path.basename(f).split("_")[-1].replace(".csv", "")
        times.add(float(t))
    return max(times)

DATA_DIR = os.path.join(TOY_DIR, "toy_output", "data")
t = latest_snapshot_time(DATA_DIR)
print(f"Plotting snapshot at t={t:.0f}s ({t/3600:.1f}h)")

background = {"co": cfg.CO_DEFAULT_MGM3, "nox": cfg.NO2_DEFAULT_MGM3, "pmx": cfg.PM2_DEFAULT_MGM3}
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, gas, label in zip(axes, ["co", "nox", "pmx"], ["CO", "NO2", "PM2.5"]):
    grid = pd.read_csv(os.path.join(DATA_DIR, f"data_{gas}_{cfg.EXPOSURE_LAYER}_{t}.csv")).values - background[gas]
    im = ax.imshow(grid.T, origin="lower", cmap="hot_r", vmin=0)
    ax.set_title(f"{label} excess [mg/m^3]")
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()


## Where to go next

- Swap in another village: copy its files from `../source_code/simulation/villages/<Village>/` into `village_data/`, and update the village name, date and file paths in `run_toy_simulation.py` (see the README).
- Change `n_people` or the category proportions in `generate_fake_population.py` to see how population assumptions affect the exposure result.
- Increase the simulated hours (up to 24) to get the full three-pollutant AQI and the diurnal pattern.
- For random local traffic and rerouting, see `run_sim.py` and `sumo_commands.py` in `../source_code/simulation/src/`.